# Notebook 00 – Research overview: Reliable CO₂ Storage Forecasting and Monitoring

**Question.** How reliably can we forecast reservoir pressure and CO₂ plume behaviour during injection and after shut-in, and can pressure and seismic observations reveal incorrect modelling assumptions?

This notebook is the entry point to the repository. It connects the question, the workflow, the existing demonstrations and the planned pilot. **It runs nothing expensive:** it uses only the Python standard library, reads files in this repository, and finishes in about a second. No simulation, training or experiment is run.

The full argument is in the [research proposal](../docs/RESEARCH_PROPOSAL.md); the first experiment is specified in the [pilot protocol](../docs/PILOT_PROTOCOL.md).

## 1. Status labels

Every claim in the repository carries one of three labels:

| Label | Meaning |
|---|---|
| **Verified result** | Obtained under a protocol frozen before the test data existed. The only verified results used here come from the completed SubsurfaceML project. |
| **Illustrative demonstration** | Runs in seconds on a textbook problem, toy data or an untrained network. It shows how a method works; its numbers are not findings. |
| **Planned experiment** | Proposed research; not done yet. |

## 2. One workflow for three applications

```
prior over reservoirs and schedules ─► reference simulator ─► verified reference results + error budget
                                                                      │
                                    forecasters with whole-trajectory bands calibrated by reservoir
                                                                      │
              ┌───────────────────────────────────────────────────────┼───────────────────────────────────┐
              ▼                                                       ▼                                   ▼
   Case A: storage assessment                      Case B: operational decisions        Case C: monitoring and model checking
   rate and duration within an assumed limit       rank schedules; simulator-confirmed  observations vs bands; alarm at a stated
                                                                                        false-alarm rate when an assumption is wrong
              ▲                                                       ▲                                   │
              └─────────────── an alarm withdraws trust in the forecasts used by A and B ◄────────────────┘
```

Each method enters where the workflow needs it. Computational mathematics verifies the reference and measures its error. Statistics and machine learning design fair tests, build forecasters and calibrate their bands. Deep learning is a candidate for map outputs, and is deferred until they are needed. Seismic methods supply a second kind of observation. See proposal §4.

## 3. What exists, and its status

The cell below lists the evidence the proposal relies on, with its label and location, and checks that every local file it points to exists.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SUBSURFACEML = "https://github.com/abdelghafarfouda/manchester-msc-projects/tree/68e532f/advanced-subsurface-modelling/SubsurfaceML"

evidence = [
    ("Verified result", "Layered CO2-brine simulator: 21 verification checks, 107 tests", SUBSURFACEML),
    ("Verified result", "Peak build-up hybrid: RMSE 0.31 MPa on 100 fresh test reservoirs (peak only, not trajectories)", SUBSURFACEML),
    ("Illustrative demonstration", "Reference checks: known answers, refinement, seeded mass-balance defect", "notebooks/01_reference_simulation_checks.ipynb"),
    ("Illustrative demonstration", "Fair evaluation design and a pressure alarm for a wrong boundary", "notebooks/02_fair_evaluation_and_boundary_alarm.ipynb"),
    ("Illustrative demonstration", "UNTRAINED surrogate architecture; outputs not physically consistent", "notebooks/03_untrained_surrogate_architecture_demo.ipynb"),
    ("Illustrative demonstration", "Seismic detectability of a thin CO2 layer", "notebooks/04_seismic_detectability.ipynb"),
    ("Planned experiment", "Pilot: pressure-based detection of a wrong outer-boundary assumption", "docs/PILOT_PROTOCOL.md"),
    ("Planned experiment", "Gravity-model convergence, seismic detectability, OPM Flow/SPE11", "docs/RESEARCH_PROPOSAL.md"),
]

missing = []
for label, what, where in evidence:
    local = not where.startswith("http")
    ok = (ROOT / where).exists() if local else True
    if not ok:
        missing.append(where)
    print(f"{label:<28} {what}\n{'':<28} -> {where}{'' if ok else '   [MISSING]'}")
print()
print("all local paths exist" if not missing else f"missing: {missing}")

Verified result              Layered CO2-brine simulator: 21 verification checks, 107 tests
                             -> https://github.com/abdelghafarfouda/manchester-msc-projects/tree/68e532f/advanced-subsurface-modelling/SubsurfaceML
Verified result              Peak build-up hybrid: RMSE 0.31 MPa on 100 fresh test reservoirs (peak only, not trajectories)
                             -> https://github.com/abdelghafarfouda/manchester-msc-projects/tree/68e532f/advanced-subsurface-modelling/SubsurfaceML
Illustrative demonstration   Reference checks: known answers, refinement, seeded mass-balance defect
                             -> notebooks/01_reference_simulation_checks.ipynb
Illustrative demonstration   Fair evaluation design and a pressure alarm for a wrong boundary
                             -> notebooks/02_fair_evaluation_and_boundary_alarm.ipynb
Illustrative demonstration   UNTRAINED surrogate architecture; outputs not physically consistent
                             ->

## 4. The supporting notebooks

The four method notebooks are supporting material. The cell below reads each one's title, and counts how often each status label appears in its text, to confirm that all four use the same three labels.

In [2]:
import json

labels = ["Illustrative demonstration", "Planned experiment", "Verified result"]
for path in sorted((ROOT / "notebooks").glob("0[1-4]_*.ipynb")):
    nb = json.loads(path.read_text(encoding="utf-8"))
    text = "".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "markdown")
    title = "".join(nb["cells"][0]["source"]).splitlines()[0].lstrip("# ")
    counts = ", ".join(f"{lab}: {text.count(lab)}" for lab in labels)
    print(f"{path.name}\n  {title}\n  {counts}")

01_reference_simulation_checks.ipynb
  Notebook 01 – Checking the reference simulation: known answers, grid refinement and mass balance
  Illustrative demonstration: 4, Planned experiment: 5, Verified result: 2
02_fair_evaluation_and_boundary_alarm.ipynb
  Notebook 02 – Fair evaluation of a fast forecaster, and a pressure alarm for a wrong boundary
  Illustrative demonstration: 8, Planned experiment: 4, Verified result: 2
03_untrained_surrogate_architecture_demo.ipynb
  Notebook 03 – Untrained surrogate architecture: structure, built-in constraints and how it would be judged
  Illustrative demonstration: 10, Planned experiment: 5, Verified result: 2
04_seismic_detectability.ipynb
  Notebook 04 – Seismic detectability of a thin CO₂ layer
  Illustrative demonstration: 5, Planned experiment: 5, Verified result: 2


Notebook 03 is an **untrained** architecture demonstration. Its outputs come from random weights. Its own checks show that they are not physically consistent: the maps hold more CO₂ than was injected, and the cumulative outflow can fall. Its §6 documents the design correction, which is not implemented. The proposal defers this network until spatial outputs justify it.

## 5. The planned pilot

**Status: Planned experiment. Not run.** Can post-shut-in pressure reveal that a forecast wrongly assumes a sealed outer boundary? Each development and test reservoir is simulated twice per schedule with the SubsurfaceML layered simulator (commit `68e532f`): sealed, so the assumption holds, and with a constant-pressure outer boundary, the controlled mismatch. Calibration reservoirs are simulated sealed only, because they set bands and alarm thresholds and nothing is fitted on them. Forecasters and detectors always assume a sealed boundary.

The cell below only does arithmetic on the planned settings of the protocol. It shows the size of the study and the false-alarm and coverage resolution that 100 test reservoirs give. It runs no simulation.

In [3]:
# Planned settings (docs/PILOT_PROTOCOL.md §5); arithmetic only, nothing is simulated
schedules = 4
roles = {  # role: (reservoirs, boundary variants, data status)
    "development (training and selection)": (220, 2, "sealed + open; existing SubsurfaceML development prior, re-run with new outputs"),
    "calibration (bands, alarm thresholds)": (100, 1, "sealed only; new seed, fixed at the freeze"),
    "independent test (scored once)": (100, 2, "sealed + open; new seed, generated after the freeze"),
}
total = 0
for role, (n_res, variants, status) in roles.items():
    runs = n_res * schedules * variants
    total += runs
    print(f"{role:<38} {n_res:>4} reservoirs x {schedules} x {variants} = {runs:>5} runs   ({status})")
print(f"{'total planned simulations':<38} {total:>31,}")

# Resolution of reservoir-level proportions with 100 test reservoirs: smallest non-zero step, and the
# conformal rank used for a 90 % band calibrated on 100 reservoirs
import math
n_cal, alpha = 100, 0.10
print(f"\nconformal rank for a 90 % band with {n_cal} calibration reservoirs: {math.ceil((n_cal + 1) * (1 - alpha))} of {n_cal}")
print("each test reservoir moves a reported coverage or false-alarm rate by 0.01")

development (training and selection)    220 reservoirs x 4 x 2 =  1760 runs   (sealed + open; existing SubsurfaceML development prior, re-run with new outputs)
calibration (bands, alarm thresholds)   100 reservoirs x 4 x 1 =   400 runs   (sealed only; new seed, fixed at the freeze)
independent test (scored once)          100 reservoirs x 4 x 2 =   800 runs   (sealed + open; new seed, generated after the freeze)
total planned simulations                                        2,960

conformal rank for a 90 % band with 100 calibration reservoirs: 91 of 100
each test reservoir moves a reported coverage or false-alarm rate by 0.01


Pilot outputs are limited to what the reference model supports: **peak pressure build-up**; **mean reservoir overpressure** at shut-in + 0.25, 1, 2 and 5 years; **pressure relaxation time**, only where relaxation occurs; and **plume radius**, which describes the layered model only. Dissolved or trapped mass, up-dip migration and CO₂ outflow are not forecast. An open boundary does not by itself produce outflow, so zero outflow is not evidence of a defect.

Before any trajectory is forecast, the protocol requires four adaptations, each with a known-answer test: recording mean pressure, monthly reporting after shut-in, documenting the shut-in pressure diagnostic as the observable, and a trajectory version of the reduced-order model. Gravity-model convergence and seismic detectability come in the next stage. OPM Flow on SPE11-type cases is a later extension.

## 6. Where to go next

- [Research proposal](../docs/RESEARCH_PROPOSAL.md): problem, gap, hypotheses, methods, evaluation, work plan, risks and references. §13 lists all evidence with its status.
- [Pilot protocol](../docs/PILOT_PROTOCOL.md): the experiment, ready to be frozen.
- Supporting notebooks: [01](01_reference_simulation_checks.ipynb) · [02](02_fair_evaluation_and_boundary_alarm.ipynb) · [03](03_untrained_surrogate_architecture_demo.ipynb) · [04](04_seismic_detectability.ipynb).

**Not claimed anywhere in this repository:** completed pilot results, field validation, validated pressure or safety thresholds, physical consistency of the untrained network, or novelty that has not been established against the literature.